# Guitar Chord Lab 04 楽曲のテンポとコードを調べる

音源を読み込んで曲全体と曲中の部分の BPM を比べます。BPM を確認したら、ギターとベースを分離し、どのコードがいつ鳴ったかを調べます。最初は、伴奏ギターがよく聴こえる短い録音で試してください。

コードの表示はメジャー、マイナー、セブンス系、パワーコードが基本です。ベース音が持続しているときは `C/E` のようなオンコードも調べます。表で「候補」と付いたものは、耳でも確かめてください。

セルを上から順に実行します。入力音源を変えるときは「2 音源を選ぶ」から、BPM を変えるときは「3 BPM を確認する」から実行し直します。

## 1 セットアップ

Colab では、講義リポジトリから Guitar Chord Lab を取得します。`REPO_REF` には、使いたいブランチ名、タグ名、コミット SHA を指定できます。この変更が `main` に入った後は、`main` も使えます。ギターの分離に時間がかかる場合は、Colab のランタイムで GPU を選びます。

YouTube を使う場合は FFmpeg と ffprobe、および Deno 2.3 以上か Node.js 22 以上が必要です。セットアップの出力で確認してください。FFmpeg がなければ、Colab のコードセルで `!apt-get -y -q install ffmpeg` を実行します。対応する JavaScript ランタイムがなければ、npm が使える環境で `!npm install -g deno` を実行してからセットアップを再確認します。 [yt-dlp の EJS ガイド](https://github.com/yt-dlp/yt-dlp/wiki/EJS) · [Deno の導入手順](https://docs.deno.com/runtime/getting_started/installation/)

ローカルで使う場合は、作品フォルダで `python -m pip install -e ".[separation,youtube]"` を実行し、この Notebook を開いてください。作品フォルダが見つからない場合は `LOCAL_PROJECT_DIR` にその場所を指定します。

In [ ]:
# @title 1 セットアップ
import os
import shutil
import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

from IPython.display import HTML, display

REPO_REF = "guitar-chord-lab-song-analysis"  # @param {type:"string"}
LOCAL_PROJECT_DIR = ""  # ローカルで作品フォルダを指定する場合に使います。
IN_COLAB = "google.colab" in sys.modules

# セットアップをやり直した場合は前回の解析結果を使いません。
session = choice = analysis = saved_dir = None
if IN_COLAB:
    repo_url = "https://github.com/shogo-ishikawa/2026_3Q_M4Th4.git"
    repo_dir = Path("/content/2026_3Q_M4Th4_guitar_song")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "--quiet", "--no-checkout", repo_url, str(repo_dir)], check=True)
    git = ["git", "-C", str(repo_dir)]
    subprocess.run(git + ["fetch", "--quiet", "origin", REPO_REF], check=True)
    commit = subprocess.check_output(git + ["rev-parse", "FETCH_HEAD"], text=True).strip()
    project_dir = repo_dir / "projects" / "guitar-chord-lab"
    loaded_package = sys.modules.get("guitar_chord_lab")
    if loaded_package is not None:
        expected = (project_dir / "src" / "guitar_chord_lab" / "__init__.py").resolve()
        if globals().get("loaded_commit") != commit or Path(loaded_package.__file__).resolve() != expected:
            raise RuntimeError("読み込んだコードの版または場所が変わります。Colab のセッションを再起動し、このセルから実行してください。")
    else:
        subprocess.run(git + ["checkout", "--quiet", "--detach", commit], check=True)
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{project_dir}[separation,youtube]"], check=True)
    loaded_commit = commit
    print("使用コミット", commit)
else:
    here = Path.cwd()
    search_dirs = [here, here.parent, here / "projects" / "guitar-chord-lab"]
    if LOCAL_PROJECT_DIR:
        search_dirs = [Path(LOCAL_PROJECT_DIR).expanduser()]
    project_dir = next((p for p in search_dirs if (p / "src" / "guitar_chord_lab").is_dir()), None)
    if project_dir is None:
        raise ValueError("LOCAL_PROJECT_DIR に Guitar Chord Lab の作品フォルダを指定してください。")
    project_dir = project_dir.resolve()

os.chdir(project_dir)
source_dir = str(project_dir / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)

from guitar_chord_lab.song import (
    NoBackingGuitar, SongCancelled, SongError,
    analyse_song, beat_rows, confirm_bpm, load_song, prepare_song, save_song,
)

print("作業フォルダ", project_dir)
for command in ("ffmpeg", "ffprobe", "deno", "node"):
    executable = shutil.which(command)
    if executable and command in ("deno", "node"):
        version = subprocess.run([executable, "--version"], capture_output=True, text=True)
        print(command, version.stdout.splitlines()[0] if version.stdout else "バージョン取得不可")
    else:
        print(command, executable or "見つかりません（YouTube 入力時の準備を確認してください）")

## 2 音源を選ぶ

`SOURCE_KIND` に `upload`、`youtube`、`local` のいずれかを選びます。

- `upload` は Colab のファイル選択から音源を1つ選びます。
- `youtube` は `YOUTUBE_URL` に動画の URL を入力します。YouTube の利用規約に沿って取得・解析できる音源に限り、`YOUTUBE_ALLOWED` にチェックを入れます。取得できなければ、許可を得て手元に用意した音声を `upload` で選べます。 [YouTube 利用規約](https://www.youtube.com/t/terms)
- `local` は、実行環境にある音声ファイルのパスを `LOCAL_AUDIO` に入力します。

`upload` と `local` では WAV、MP3、FLAC、OGG を使えます。取得した音源と Colab にアップロードしたコピーは、読み込み後に一時フォルダから取り除かれます。元の手元のファイルと、後で保存する解析結果は別に管理します。

In [ ]:
# @title 2 音源を選ぶ
SOURCE_KIND = "upload"  # @param ["upload", "youtube", "local"]
YOUTUBE_URL = ""  # @param {type:"string"}
YOUTUBE_ALLOWED = False  # @param {type:"boolean"}
LOCAL_AUDIO = ""  # @param {type:"string"}

session = choice = analysis = saved_dir = None
try:
    if SOURCE_KIND == "upload":
        if not IN_COLAB:
            raise ValueError("ローカルの Notebook では SOURCE_KIND を local にしてください。")
        from google.colab import files

        with TemporaryDirectory(prefix="guitar-chord-upload-") as folder:
            uploaded = files.upload(target_dir=folder)
            if len(uploaded) != 1:
                raise ValueError("音声ファイルを1つ選んでください。")
            path = Path(next(iter(uploaded)))
            del uploaded
            session = prepare_song(Path(folder) / path)
    elif SOURCE_KIND == "youtube":
        if not YOUTUBE_URL.strip():
            raise ValueError("YOUTUBE_URL に動画の URL を入れてください。")
        session = prepare_song(YOUTUBE_URL.strip(), youtube_allowed=YOUTUBE_ALLOWED)
    elif SOURCE_KIND == "local":
        if not LOCAL_AUDIO.strip():
            raise ValueError("LOCAL_AUDIO に音声ファイルのパスを入れてください。")
        session = prepare_song(Path(LOCAL_AUDIO).expanduser())
    else:
        raise ValueError("SOURCE_KIND は upload、youtube、local のいずれかにしてください。")
except (SongError, ValueError, OSError) as error:
    print(f"音源を準備できませんでした。{error}")

if session is not None:
    print(f"曲の推定区間 {session.report.start_s:.2f}〜{session.report.end_s:.2f} 秒")
    print("次のセルで BPM を確認します。")

## 3 BPM を確認する

全体の推定値と、曲中の複数区間で推定した値を見比べます。区間ごとの値は、極端な値の影響を受けにくい**中央値**でも確認します。BPM が半分や倍に聞こえることや、曲の途中でテンポが変わることがあります。

入力欄で `A`（全体 BPM を採用）、`B`（再推定）、`C`（手動入力）、`Q`（中止）を選びます。手動入力は20〜360の範囲で、`123.5` のような小数も使えます。推定値を採用した場合は音源から検出した拍位置を優先し、手動入力した場合は指定した BPM の間隔で拍を配置します。

In [ ]:
# @title 3 BPM を確認する
choice = analysis = saved_dir = None
if session is None:
    print("先に「2 音源を選ぶ」を実行してください。")
else:
    try:
        choice = confirm_bpm(session)
    except SongCancelled as error:
        print(error)
    except SongError as error:
        print(f"BPM を決定できませんでした。{error}")
    if choice is not None:
        print(f"採用した BPM {choice.bpm:g}")

## 4 分離してコードを解析する

分離されたギターから、持続する複数音などを手掛かりに伴奏を探します。伴奏の根拠が足りなければ、理由を表示してコード解析を止めます。ギターの音量が小さい場合や単音の伴奏では、判定が難しくなる場合があります。リードと伴奏が同じギターステムに残る場合もあります。

分離モデルの取得や分離処理に失敗した場合は、原音から候補を推定する経路に切り替わり、その旨を表示します。

小節表示は既定で4拍です。拍子の候補が得られた場合は、表示上の拍数と併せて表示します。実際の拍子が分かっている場合は `BEATS_PER_BAR` を変更してください。小節番号は推定した拍を起点とする目安です。

In [ ]:
# @title 4 分離してコードを解析する
DEVICE = "auto"  # @param ["auto", "cpu", "cuda"]
BEATS_PER_BAR = 4  # @param {type:"integer"}

analysis = saved_dir = None
if session is None or choice is None:
    print("「2 音源を選ぶ」と「3 BPM を確認する」を実行してください。")
else:
    try:
        analysis = analyse_song(session, choice, device=DEVICE, meter=BEATS_PER_BAR)
    except NoBackingGuitar as error:
        print(error)
    except (SongError, ValueError, OSError) as error:
        print(f"解析を完了できませんでした。{error}")

    if analysis is not None:
        separation = analysis.metadata["separation"]
        if separation["status"] == "fallback":
            print("分離処理に失敗したため、原音から候補を推定しました。")
            print(separation["reason"])
        elif separation["harmony_note"]:
            print(separation["harmony_note"])
        tempo = analysis.metadata["tempo"]
        if tempo["meter"] is not None:
            print(f"拍の強さから見た拍子の候補 {tempo['meter']}拍子")
        print(f"表示上の1小節 {analysis.metadata['beats_per_bar']}拍")
        print("次のセルで、拍ごとの結果を表示します。")

## 5 コードを拍ごとに読む

表の各行が1小節です。拍の途中でコードが変わった場合は、その拍の欄に位置と推定時刻も表示します。`N` は、その区間で採用できるコード名や候補を得られなかったことを表します。

例えば BPM が120なら1拍は0.5秒で、4分の1拍は0.125秒です。変化を探す時間の細かさの目安に使います。コード判定を0.125秒ごとに区切ったり、変化が必ずその位置に起きたと見なしたりはしません。

In [ ]:
# @title 5 コードを拍ごとに読む
from html import escape

def show_beat_table(result):
    rows = beat_rows(result)
    meter = result.metadata["beats_per_bar"]
    measures = {}
    for row in rows:
        bar, beat = row["bar"], row["beat"]
        chord = escape(str(row["chord"]))
        times = f"{row['start_s']:.2f}〜{row['end_s']:.2f} 秒"
        entry = f"<strong>{chord}</strong><br><small>{times}</small>"
        if row["changes"]:
            entry += f"<br><small>{escape(str(row['changes']))}</small>"
        measures.setdefault(bar, {})[beat] = entry

    headers = "".join(f"<th scope='col'>{beat}拍目</th>" for beat in range(1, meter + 1))
    table = [f"<table><thead><tr><th scope='col'>小節</th>{headers}</tr></thead><tbody>"]
    for bar, entries in measures.items():
        beats = "".join(f"<td>{entries.get(beat, '—')}</td>" for beat in range(1, meter + 1))
        table.append(f"<tr><th scope='row'>{bar}</th>{beats}</tr>")
    table.append("</tbody></table>")
    display(HTML("".join(table)))

if analysis is None:
    print("表示できる今回の解析結果がありません。前のセルの表示を確認してください。")
else:
    show_beat_table(analysis)

## 6 結果を保存する

`chords.csv` はコードの種類、`timing.csv` は変化時刻、`judgements.csv` は各区間の判定です。コード ID とチェンジ ID で表を関連付けられます。各表は JSON でも保存します。`song.json` には解析条件、BPM、曲の推定区間、分離の記録が入ります。

Colab のセッション終了後も結果を使う場合は、`DOWNLOAD` にチェックを入れて ZIP をダウンロードします。ZIP には今回の解析結果が入ります。

In [ ]:
# @title 6 結果を保存する
DOWNLOAD = False  # @param {type:"boolean"}
if analysis is None:
    print("保存できる今回の解析結果がありません。先に「4 分離してコードを解析する」を実行してください。")
else:
    if saved_dir is None or not saved_dir.is_dir():
        saved_dir = save_song(analysis, Path("outputs") / "song_analysis")
    print("保存先", saved_dir.resolve())
    for path in sorted(saved_dir.iterdir()):
        if path.is_file():
            print(" ", path.name)
    if DOWNLOAD:
        zip_path = shutil.make_archive(str(saved_dir), "zip", root_dir=saved_dir.parent, base_dir=saved_dir.name)
        if IN_COLAB:
            from google.colab import files

            files.download(zip_path)
        else:
            print("ZIP", Path(zip_path).resolve())

## 7 保存した結果を読み直す（任意）

保存後に `RELOAD` にチェックを入れると、音源を読み込まず、保存した JSON から同じ拍の表を表示できます。前のセッションの結果を使う場合は、ダウンロードした ZIP を解凍し、`RESULT_DIR` に `song.json` のあるフォルダを指定します。表の表示には「1 セットアップ」と「5 コードを拍ごとに読む」のセルを先に実行してください。

In [ ]:
# @title 7 保存した結果を読み直す（任意）
RELOAD = False  # @param {type:"boolean"}
RESULT_DIR = ""  # @param {type:"string"}

if RELOAD:
    folder = Path(RESULT_DIR.strip()).expanduser() if RESULT_DIR.strip() else saved_dir
    if folder is None:
        print("先に保存するか、RESULT_DIR に保存先のフォルダを指定してください。")
    else:
        try:
            restored = load_song(folder)
            print("読み込んだ解析 ID", restored.metadata["analysis_id"])
            show_beat_table(restored)
        except (ValueError, OSError) as error:
            print(f"保存結果を読み込めませんでした。{error}")

## 観察してみよう

1. `C` と `C5` の構成音を比べてください。C と G がはっきり聴こえ、E や E♭ の根拠がないとき、どちらの判定が妥当か考えます。
2. 拍の途中で切り替わった箇所を1つ選び、元の録音と照らし合わせてください。CSV の推定時刻と拍の目安を別々に記録します。
3. 「候補」や `N` の箇所を1つ選び、音域、ベース、音の持続、リードギターの重なりのうち何が影響したか考えてください。推定スコアと正解率は別の値です。